# 17.1 Learning Rate

## Concept
The learning rate is the size of the step the model takes when it updates its weights after each mistake. A small step learns slowly; a big step can jump past the best value.

## Why it is required / what problem it solves
It is the most sensitive hyperparameter. A wrong value means the model either barely learns or blows up, so it must be chosen and tested first.

In [1]:
# Minimize loss = (w - 3)^2 ; the best weight is w = 3
def train(lr, steps=20):
    w = 0.0
    for _ in range(steps):
        grad = 2 * (w - 3)      # slope of the loss
        w = w - lr * grad       # learning rate controls the step size
    return w, (w - 3) ** 2

for lr in [0.01, 0.1, 0.5, 1.1]:
    w, loss = train(lr)
    print(f"lr={lr:<5} final w={w:10.4f}  loss={loss:.4f}")

lr=0.01  final w=    0.9972  loss=4.0113
lr=0.1   final w=    2.9654  loss=0.0012
lr=0.5   final w=    3.0000  loss=0.0000
lr=1.1   final w= -112.0128  loss=13227.9441


## What this example tells us
- lr=0.01 is too small: after 20 steps the weight is still far from the best value (3).
- lr=0.1 and lr=0.5 reach the best weight quickly.
- lr=1.1 is too big: the weight overshoots, moves further away each step, and the loss explodes.

## AI/ML Example
In the digits network (Notebook 17), a learning rate of 0.001 gave about 41% accuracy, while 0.1 gave about 96%.

## Business Example
A sales forecasting model trained with a too-large learning rate gives wild, useless predictions; a well-chosen one gives stable forecasts and saves retraining time and cost.

## AI/ML Engineer Use Case
An engineer tries several learning rates first (for example 0.1, 0.01, 0.001), compares validation loss, and keeps the one that decreases smoothly without exploding.

# 17.2 Batch Size

## Concept
Batch size is the number of training samples the model looks at before it updates its weights once.

## Why it is required / what problem it solves
It controls how often weights are updated, how noisy each update is, and how much memory and time training needs. Real datasets are too big to process in one go, so a batch size must be chosen.

In [2]:
import numpy as np

# Simple data: y = 2x + 1 with a little noise (100 samples)
rng = np.random.RandomState(0)
X = rng.rand(100)
y = 2 * X + 1 + rng.randn(100) * 0.1

def train(batch_size, epochs=5, lr=0.1):
    w, b, updates = 0.0, 0.0, 0
    for _ in range(epochs):
        idx = rng.permutation(100)
        for s in range(0, 100, batch_size):          # one batch at a time
            xb, yb = X[idx[s:s + batch_size]], y[idx[s:s + batch_size]]
            err = (w * xb + b) - yb
            w -= lr * 2 * (err * xb).mean()          # update after EACH batch
            b -= lr * 2 * err.mean()
            updates += 1
    loss = ((w * X + b - y) ** 2).mean()
    return updates, loss

for bs in [1, 10, 100]:
    updates, loss = train(bs)
    print(f"batch_size={bs:<4} weight updates={updates:<4} final loss={loss:.4f}")

batch_size=1    weight updates=500  final loss=0.0131
batch_size=10   weight updates=50   final loss=0.0397
batch_size=100  weight updates=5    final loss=0.3473


## What this example tells us
- With 100 samples, batch size 1 gives 500 updates in 5 epochs, batch size 10 gives 50, and batch size 100 gives only 5.
- More updates per epoch meant a lower loss here (0.0131 vs 0.3473) after the same number of epochs.
- Very small batches are noisy and slow on a GPU; very large batches are fast per epoch but learn less per epoch. A middle value (16, 32, 64) is the usual choice.

## AI/ML Example
In the digits network (Notebook 17), batch size 8 gave 96.94% accuracy and batch size 128 gave 93.04% with the same epochs.

## Business Example
A retail company training a demand model on millions of rows picks a batch size that fits GPU memory and finishes overnight, keeping cloud cost under control.

## AI/ML Engineer Use Case
An engineer sets the largest batch size that fits in GPU memory, then checks that validation accuracy does not drop, and adjusts the learning rate if the batch size changes a lot.

# 17.3 Number of Epochs

## Concept
One epoch is one full pass of the model over the whole training data. The number of epochs is how many times the model repeats this.

## Why it is required / what problem it solves
One pass is rarely enough to learn. Too few epochs leave the model under-trained; too many waste time and can make it memorize the training data (overfitting).

In [3]:
import numpy as np

# Same simple data: y = 2x + 1 with a little noise
rng = np.random.RandomState(0)
X = rng.rand(100)
y = 2 * X + 1 + rng.randn(100) * 0.1

w, b, lr = 0.0, 0.0, 0.1
for epoch in range(1, 101):                          # one epoch = one full pass over the data
    err = (w * X + b) - y
    w -= lr * 2 * (err * X).mean()
    b -= lr * 2 * err.mean()
    if epoch in [1, 5, 20, 100]:
        loss = ((w * X + b - y) ** 2).mean()
        print(f"epochs={epoch:<4} w={w:.3f} b={b:.3f} loss={loss:.4f}")

epochs=1    w=0.219 b=0.393 loss=2.4278
epochs=5    w=0.699 b=1.189 loss=0.3473
epochs=20   w=1.095 b=1.471 loss=0.0777
epochs=100  w=1.689 b=1.176 loss=0.0177


## What this example tells us
- The loss fell from 2.4278 (1 epoch) to 0.3473 (5), 0.0777 (20) and 0.0177 (100).
- Early epochs give big improvements; later epochs give smaller and smaller gains.
- In real projects we watch the validation loss and stop when it stops improving.

## AI/ML Example
In the digits network (Notebook 17), 5 epochs gave 89.97% accuracy, 30 epochs gave 95.82% and 80 epochs gave 96.66%.

## Business Example
A bank training a fraud model stops training once accuracy stops improving, saving compute cost instead of running for days with no benefit.

## AI/ML Engineer Use Case
An engineer uses early stopping: train for many epochs, monitor validation loss, and keep the model from the epoch where validation loss was lowest.

# 17.4 Number of Hidden Layers

## Concept
Hidden layers are the layers between the input and the output of a neural network. Each extra layer lets the network build more complex patterns on top of the previous layer.

## Why it is required / what problem it solves
With no hidden layer the model can only draw a straight line. Hidden layers let it learn curved and complex patterns, but more layers cost more time and can overfit.

In [4]:
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

# Circular data: one class inside a ring of the other (a straight line cannot separate it)
X, y = make_circles(n_samples=600, noise=0.1, factor=0.4, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

models = {
    "0 hidden layers (straight line)": LogisticRegression(),
    "1 hidden layer  (8)": MLPClassifier((8,), max_iter=3000, random_state=0),
    "2 hidden layers (8, 8)": MLPClassifier((8, 8), max_iter=3000, random_state=0),
}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    print(f"{name:<32} test accuracy = {model.score(X_te, y_te):.3f}")

0 hidden layers (straight line)  test accuracy = 0.544
1 hidden layer  (8)              test accuracy = 0.989
2 hidden layers (8, 8)           test accuracy = 0.989


## What this example tells us
- With no hidden layer (a straight line), test accuracy was only 54.4% on the circular data.
- One hidden layer jumped to 98.9%, because it can bend the decision boundary.
- A second layer gave the same accuracy here, so extra depth was not needed for this simple data.

## AI/ML Example
In the digits network (Notebook 17), 1 to 3 hidden layers all gave about 95% to 97%, so a small network was enough.

## Business Example
A telecom company predicting customer churn finds that a model with one or two hidden layers is as accurate as a deep one, so it ships the simpler, cheaper model.

## AI/ML Engineer Use Case
An engineer starts with 1 or 2 hidden layers and adds more only if validation accuracy clearly improves.

# 17.5 Number of Neurons

## Concept
Neurons are the small units inside a layer. The number of neurons in a layer decides how much the layer can learn (its width).

## Why it is required / what problem it solves
Too few neurons cannot capture the pattern (underfitting). Too many add cost and can memorize the training data, so the right size must be found by testing.

In [5]:
from sklearn.datasets import make_circles
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

X, y = make_circles(n_samples=600, noise=0.1, factor=0.4, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

for n in [1, 2, 4, 16, 64]:                          # neurons in the single hidden layer
    model = MLPClassifier((n,), max_iter=3000, random_state=0).fit(X_tr, y_tr)
    print(f"neurons={n:<3} train acc={model.score(X_tr, y_tr):.3f}  test acc={model.score(X_te, y_te):.3f}")

neurons=1   train acc=0.440  test acc=0.456
neurons=2   train acc=0.798  test acc=0.811
neurons=4   train acc=0.993  test acc=0.972
neurons=16  train acc=0.998  test acc=1.000
neurons=64  train acc=1.000  test acc=0.994


## What this example tells us
- With 1 neuron, test accuracy was 45.6%, and with 2 neurons 81.1%: too small to learn the circular pattern.
- With 4 neurons it reached 97.2%, and with 16 neurons 100%.
- With 64 neurons it was 99.4%: no further gain, only more compute.

## AI/ML Example
In the digits network (Notebook 17), 8 neurons gave 96.38% and 128 neurons gave 95.26%, so bigger was not better.

## Business Example
A logistics company uses a model with just enough neurons to predict delivery time accurately, keeping the model fast and cheap to run on its servers.

## AI/ML Engineer Use Case
An engineer tries widths such as 16, 32, 64, 128, and picks the smallest one whose validation score is close to the best.

# 17.6 Dropout

## Concept
Dropout randomly switches off a fraction of neurons during training. Each training step uses a slightly different smaller network, so no neuron can depend too much on others.

## Why it is required / what problem it solves
It is a simple and effective way to reduce overfitting, which is when a model does very well on training data but poorly on new data. Dropout is turned off when making real predictions.

In [6]:
import numpy as np

rng = np.random.RandomState(3)
activations = np.array([0.8, 1.2, 0.5, 2.0, 1.5, 0.9, 1.1, 0.7])   # outputs of 8 neurons
rate = 0.5                                                          # drop 50% of neurons

mask = rng.rand(8) > rate                    # 1 = keep, 0 = switch off (random every time)
dropped = activations * mask / (1 - rate)    # scale up the kept ones to compensate

print("original :", activations)
print("mask     :", mask.astype(int))
print("after    :", dropped)

# Over many random masks the average output matches the original
avg = np.mean([activations * (rng.rand(8) > rate) / (1 - rate) for _ in range(10000)], axis=0)
print("average over 10000 masks:", avg.round(2))

original : [0.8 1.2 0.5 2.  1.5 0.9 1.1 0.7]
mask     : [1 1 0 1 1 1 0 0]
after    : [1.6 2.4 0.  4.  3.  1.8 0.  0. ]
average over 10000 masks: [0.8  1.2  0.5  2.03 1.51 0.9  1.1  0.7 ]


## What this example tells us
- The mask shows which neurons stay (1) and which are dropped (0) in one step; dropped outputs become 0.
- Kept neurons are scaled by 1/(1 - rate) = 2, so the overall signal stays about the same.
- Over 10000 random masks, the average output matches the original values, so dropout does not change what the network learns on average.

## AI/ML Example
In the digits network (Notebook 17), a 128-neuron network scored 95.26% without dropout and 97.21% with dropout 0.2.

## Business Example
A hospital model trained on a small patient dataset uses dropout so it works well on new patients and not only on the ones it has already seen.

## AI/ML Engineer Use Case
An engineer adds dropout (usually 0.1 to 0.5) when training accuracy is much higher than validation accuracy, and lowers it if the model starts underfitting.

# 17.7 Optimizer

## Concept
The optimizer is the method that decides how to update the weights using the gradients. Common ones are SGD, Momentum and Adam.

## Why it is required / what problem it solves
Different optimizers reach a good solution at different speeds. A good optimizer trains faster and more reliably, especially when the loss surface is steep in one direction and flat in another.

In [7]:
import numpy as np

# Bowl-shaped loss: very flat in x, steep in y. Best point is (0, 0).
def loss(p):  return 0.1 * p[0] ** 2 + 5 * p[1] ** 2
def grad(p):  return np.array([0.2 * p[0], 10 * p[1]])

def run(optimizer, lr, steps=50):
    p = np.array([5.0, 5.0]); m = np.zeros(2); v = np.zeros(2)
    for t in range(1, steps + 1):
        g = grad(p)
        if optimizer == "SGD":
            p = p - lr * g
        elif optimizer == "Momentum":                 # remembers the previous direction
            m = 0.9 * m - lr * g; p = p + m
        else:                                         # Adam: momentum + per-weight step size
            m = 0.9 * m + 0.1 * g; v = 0.999 * v + 0.001 * g ** 2
            p = p - lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
    return loss(p)

for name, lr in [("SGD", 0.1), ("Momentum", 0.02), ("Adam", 0.5)]:
    print(f"{name:<9} lr={lr:<5} loss after 50 steps = {run(name, lr):.6f}")

SGD       lr=0.1   loss after 50 steps = 0.331549
Momentum  lr=0.02  loss after 50 steps = 0.127689
Adam      lr=0.5   loss after 50 steps = 0.002960


## What this example tells us
- After 50 steps, SGD reached a loss of 0.3315, Momentum 0.1277 and Adam 0.0030.
- Momentum remembers its earlier direction, so it moves faster along the flat direction.
- Adam adapts the step size for each weight, so it handled the steep and flat directions best here.
- Each optimizer used its own suitable learning rate, and this is a small toy problem, so results on real data can differ.

## AI/ML Example
In the digits network (Notebook 17), SGD, Momentum and Adam all reached about 96% once their learning rates were tuned.

## Business Example
A company retraining a recommendation model every night uses Adam because it converges quickly with little tuning, finishing before the next business day.

## AI/ML Engineer Use Case
An engineer usually starts with Adam (learning rate around 0.001) as a safe default, and tries SGD with momentum when fine-tuning for the best final accuracy.

# 17.8 Activation Function

## Concept
An activation function is applied to a neuron's output. It decides how strongly the neuron passes its signal on. Common ones are ReLU, Sigmoid and Tanh.

## Why it is required / what problem it solves
Without activation functions, a network of many layers is still just a straight line. Activations add the non-linearity that lets the network learn complex patterns.

In [8]:
import numpy as np

x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

relu    = np.maximum(0, x)                  # negative -> 0, positive stays
sigmoid = 1 / (1 + np.exp(-x))              # squashes to 0..1
tanh    = np.tanh(x)                        # squashes to -1..1

print("input   :", x)
print("ReLU    :", relu)
print("Sigmoid :", sigmoid.round(3))
print("Tanh    :", tanh.round(3))

input   : [-2. -1.  0.  1.  2.]
ReLU    : [0. 0. 0. 1. 2.]
Sigmoid : [0.119 0.269 0.5   0.731 0.881]
Tanh    : [-0.964 -0.762  0.     0.762  0.964]


## What this example tells us
- ReLU turns negative values into 0 and keeps positive values: simple and fast.
- Sigmoid squeezes values between 0 and 1, which is useful for probabilities in the output layer.
- Tanh squeezes values between -1 and 1 and is centered at 0.

## AI/ML Example
In the digits network (Notebook 17), ReLU gave 95.82%, Tanh 95.26% and Sigmoid 90.81%.

## Business Example
A credit-risk model uses ReLU inside the network and a Sigmoid at the end to output the probability that a customer will default.

## AI/ML Engineer Use Case
An engineer uses ReLU for hidden layers by default, Sigmoid for yes/no outputs, and Softmax for multi-class outputs.

# 17.9 Weight Decay

## Concept
Weight decay adds a small penalty for large weights during training (an L2 penalty). It pushes the model to keep its weights small and simple.

## Why it is required / what problem it solves
Models that overfit often use very large weights to fit noise in the training data. Weight decay keeps weights small, so the model generalizes better to new data.

In [9]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# Small noisy data from a simple curve; a degree-12 model can easily overfit it
rng = np.random.RandomState(0)
X = rng.uniform(-1, 1, (40, 1))
y = np.sin(2 * X[:, 0]) + rng.randn(40) * 0.2
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)

for alpha in [0.0, 0.001, 0.1]:                  # alpha = weight decay strength (L2 penalty)
    model = make_pipeline(PolynomialFeatures(12), Ridge(alpha=alpha)).fit(X_tr, y_tr)
    weights = model[-1].coef_
    train_err = ((model.predict(X_tr) - y_tr) ** 2).mean()
    test_err = ((model.predict(X_te) - y_te) ** 2).mean()
    print(f"weight decay={alpha:<6} biggest |weight|={np.abs(weights).max():9.1f}  train error={train_err:.4f}  test error={test_err:.4f}")

weight decay=0.0    biggest |weight|=   2418.9  train error=0.0110  test error=0.2589
weight decay=0.001  biggest |weight|=      2.7  train error=0.0195  test error=0.1801
weight decay=0.1    biggest |weight|=      1.4  train error=0.0270  test error=0.0436


## What this example tells us
- With no weight decay, the biggest weight was 2418.9 and the test error was high (0.2589), even though the train error was very low (0.0110): classic overfitting.
- With weight decay 0.001 the biggest weight fell to 2.7 and the test error to 0.1801.
- With weight decay 0.1 the test error was only 0.0436, as the train error rose slightly (0.0270).
- Too much weight decay can underfit; in the digits network (Notebook 17), 1e-1 dropped accuracy to 77.44%.

## AI/ML Example
Large neural networks are almost always trained with a small weight decay, such as 1e-4 or 1e-2 with AdamW, to improve results on unseen data.

## Business Example
A price-prediction model with many input features uses weight decay so it does not chase noise in historical data and gives steadier price estimates.

## AI/ML Engineer Use Case
An engineer tries a few values (for example 0, 1e-5, 1e-4, 1e-3) and picks the one with the best validation score, checking that training accuracy is not hurt too much.